# Data residency: agent의 inference 지역 고정하기

규제 대상 데이터를 다루는 agent는 출시 전에 한 가지 질문에 답해야 합니다. model 요청은 어디에서 실행되는가입니다. proxy나 runbook에서 답하면 이를 강제하는 시스템 밖에 보장이 놓입니다.

`model.inference_geo`는 답을 agent 정의에 둡니다. agent의 model 요청을 처리하는 지역을 고정하고 workspace의 residency 정책에 따라 검증하며 모든 turn에서 적용하므로, 제약이 agent와 함께 유지되고 출시 뒤 정책이 바뀌어도 유효합니다.

이 notebook에서는 다음을 살펴봅니다.

- agent를 `us` inference로 고정하고 세션에서 확인하기
- workspace의 `allowed_inference_geos` 및 `default_inference_geo`와 고정값의 관계
- 고정값을 해제하는 update 동작
- `agent_with_overrides`로 세션 하나의 지역 덮어쓰기


## 1. client 설정하기

`inference_geo`는 표준 `managed-agents-2026-04-01` beta header를 사용하며 타입이 지정된 필드를 사용하려면 `anthropic>=0.121.0`이 필요합니다.


In [ ]:
%%capture
%pip install -qU "anthropic>=0.121.0" python-dotenv

In [ ]:
import os

import anthropic
from dotenv import load_dotenv

load_dotenv()

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-5")
client = anthropic.Anthropic()

## 2. agent를 US inference로 고정하기

`inference_geo`는 `model` 설정에서 `effort`, `speed`와 함께 있습니다. 허용 값은 `"global"`과 `"us"`입니다. 생략하면 모든 model 요청은 처리되는 시점의 workspace `default_inference_geo`를 따릅니다. 즉, 나중에 workspace 기본값을 바꾸면 이미 실행 중인 세션에도 적용됩니다. 설정하면 agent의 지역이 고정됩니다.

고정값은 세 번 검증됩니다. agent 저장 시, agent에서 세션 생성 시, 세션이 처리하는 모든 turn에서입니다. 기존 설정을 예외로 인정하지 않고 엄격하게 적용됩니다. workspace 관리자가 나중에 `allowed_inference_geos`를 좁히면 이제 허용되지 않는 agent의 새 세션이 거부되고, 이미 실행 중인 세션은 다음 turn을 거부합니다. 이 엄격함이 목적입니다. workspace는 residency를 위해 이를 의존하므로 세션 도중 정책이 바뀌어도 고정 제약은 유지됩니다.


In [ ]:
researcher = client.beta.agents.create(
    name="us_records_analyst",
    description="Answers questions about internal records, pinned to US inference.",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-pin-inference-geo"},
    model={"id": MODEL, "inference_geo": "us"},
    system="You answer questions about the records you are given, concisely.",
    tools=[{"type": "agent_toolset_20260401"}],
    betas=BETAS,
)
print(f"{researcher.name}: {researcher.id} v{researcher.version}")
print("inference_geo:", researcher.model.inference_geo)

## 3. 세션에서 고정값 확인하기

세션은 생성 시 agent 설정의 snapshot을 보관하므로 `session.agent.model`에서 고정값을 볼 수 있습니다. 이를 다시 읽으면 현재 workspace 기본값과 관계없이 이 세션의 turn이 `us`에 묶여 있는지 확인할 수 있습니다.


In [ ]:
env = client.beta.environments.create(
    name="residency-demo",
    config={"type": "anthropic_cloud", "networking": {"type": "unrestricted"}},
    betas=BETAS,
)

session = client.beta.sessions.create(
    agent=researcher.id,
    environment_id=env.id,
    title="Records question, US-pinned",
    initial_events=[
        {
            "type": "user.message",
            "content": [
                {"type": "text", "text": "In one sentence: what is a data residency policy?"}
            ],
        }
    ],
    betas=BETAS,
)
print("session pin:", session.agent.model.inference_geo)

with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    for ev in stream:
        if ev.type == "agent.message":
            print("".join(b.text for b in ev.content if b.type == "text"), end="")
        elif ev.type == "session.status_idle":
            print(f"\n[idle] stop_reason={ev.stop_reason.type}")
            break

## 4. workspace가 고정값이 가리킬 수 있는 지역 결정하기

workspace의 residency 정책은 Admin API에서 workspace의 `data_residency` block에 있습니다. `allowed_inference_geos`(지역 목록 또는 리터럴 `"unrestricted"`)와 `default_inference_geo`가 포함됩니다. 해당 필드가 `"unrestricted"`가 아니라면 agent의 `inference_geo`는 `allowed_inference_geos`에 속해야 합니다. 그 외에는 저장 시 400입니다. multiagent roster에서는 모두 일치해야 합니다. coordinator와 모든 roster 구성원이 같은 지역으로 설정되어 있거나 모두 설정되어 있지 않아야 합니다.

명확히 알아 둘 update의 미묘한 점이 하나 있습니다. `agents.update`에서 `model`은 merge가 아니라 통째로 교체되므로 `inference_geo` 없이 `model`을 보내면 고정값이 해제됩니다. model id를 변경하면서 지역을 유지하려면 다시 명시하세요.


In [ ]:
cleared = client.beta.agents.update(researcher.id, model={"id": MODEL}, betas=BETAS)
print("after model update without geo:", cleared.model.inference_geo)

researcher = client.beta.agents.update(
    researcher.id,
    model={"id": MODEL, "inference_geo": "us"},  # 고정값 다시 명시
    betas=BETAS,
)
print("restated:", researcher.model.inference_geo)

## 5. 세션 하나의 지역 덮어쓰기

agent의 고정값은 agent에서 생성하는 모든 세션의 기본값입니다. 공유 agent를 편집하지 않고 한 세션을 다른 지역에서 실행하려면 `sessions.create`에 `agent_with_overrides` 객체를 전달하고 `model`을 덮어씁니다. override는 해당 세션의 model 설정만 교체하며 agent resource는 변경되지 않습니다. 같은 소속 규칙이 적용됩니다. override한 지역은 workspace에서 허용되어야 하며 roster 구성원도 여전히 모두 일치해야 합니다.


In [ ]:
override = client.beta.sessions.create(
    agent={
        "type": "agent_with_overrides",
        "id": researcher.id,
        "model": {"id": MODEL, "inference_geo": "global"},  # 이 세션에만 적용
    },
    environment_id=env.id,
    title="One-off, global geography",
    betas=BETAS,
)
print("override session pin:", override.agent.model.inference_geo)
print(
    "agent still pinned to:",
    client.beta.agents.retrieve(researcher.id, betas=BETAS).model.inference_geo,
)

## 6. 정리하기


In [ ]:
from utilities import wait_for_idle_status

wait_for_idle_status(client, session.id)
for s in (session, override):
    client.beta.sessions.archive(s.id, betas=BETAS)
client.beta.agents.archive(researcher.id, betas=BETAS)
client.beta.environments.archive(env.id, betas=BETAS)
print("archived")

## 지역 고정을 활용할 곳

`inference_geo`는 residency와 compliance 제어 수단입니다. agent가 무엇을 읽을 수 있는지 또는 데이터가 어디에 저장되는지는 나타내지 않으며, token이 어디에서 처리되는지만 나타냅니다. 저장 지역은 별도의 변경 불가능한 workspace 설정(`workspace_geo`)이며, 둘이 함께 compliance review의 질문에 답합니다.

생성 시에만 아니라 모든 turn에서 적용되므로 agent 출시 후 변경에도 보장이 유지되어야 할 때 적합한 제어 수단입니다. 여러 agent를 운영한다면 workspace의 `default_inference_geo`를 설정하여 고정되지 않은 agent가 적절한 지역을 상속하게 하고, residency가 그 기본값에 절대 의존해서는 안 되는 agent에만 agent별 고정값을 사용하세요.
